# Spotify XAI Analysis

End-to-end pipeline: EDA → Modeling → Explainability (PDP/ICE)

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm joblib tabulate nbformat ipykernel


## Section 0: Setup

In [ ]:
import sys
import os

sys.path.insert(0, '../src')

import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt

from utils import ensure_dirs, set_plot_style, print_section
from preprocessing import (
    load_and_clean, distribution_analysis,
    correlation_with_popularity, bias_check, engineer_features,
)
from modeling import (
    split_data, train_all_models, evaluate_models,
    run_isolation_forest, save_models,
)
from pdp_ice import (
    select_pdp_features,
    plot_pdp_grid,
    plot_ice_grid,
    pdp_ice_interpretation_table,
)

ensure_dirs('../outputs')
set_plot_style()
print("Setup complete.")


## Section 1: Phase 1 — Data Loading and EDA

### 1.1 Load and Clean

In [ ]:
df = load_and_clean('../dataset.csv')

### 1.2 Distribution Analysis

Histograms and boxplots reveal the spread and skewness of each audio feature. Highly skewed features (|skew| > 1) may need transformation for linear models, though tree-based models are robust to skew.

In [ ]:
distribution_analysis(df, '../outputs/figures')

### 1.3 Correlation with Popularity

Pearson correlation shows linear relationships. Loudness and energy tend to have the strongest positive correlations with popularity — louder, energetic tracks are associated with mainstream appeal. Acousticness and instrumentalness often show negative correlation, reflecting a preference for vocal, produced sound in popular music.

In [ ]:
correlation_with_popularity(df, '../outputs/figures')

### 1.4 Bias Check

Uneven genre representation can bias model predictions toward over-represented genres. Artists with many tracks may also skew the learned relationships between audio features and popularity.

In [ ]:
bias_report = bias_check(df, '../outputs/figures')
print(bias_report)

## Section 2: Phase 2 — Feature Engineering and Modeling

### 2.1 Feature Engineering

In [ ]:
X, y, feature_names = engineer_features(df)
print(f"Features: {len(feature_names)}")
print(X.dtypes.value_counts())


### 2.2 Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

### 2.3 Train All Models

We train four models: a linear baseline (LinearRegression), two ensemble tree models (RandomForest, XGBoost), and a gradient boosting model (LightGBM). Tree-based models are expected to capture non-linear audio feature interactions.

In [ ]:
models_dict = train_all_models(X_train, y_train)

### 2.4 Evaluate Models

RMSE measures prediction error in popularity units (0–100). R² indicates how much variance in popularity the model explains. Popularity is notoriously hard to predict from audio features alone — values of R² ~ 0.2–0.4 are typical for audio-only models.

In [ ]:
results_df = evaluate_models(models_dict, X_test, y_test)
results_df

### 2.5 Anomaly Detection — Isolation Forest

Isolation Forest flags tracks whose audio feature combinations are unusual compared to the rest of the dataset. These may be mislabelled tracks, niche experimental recordings, or data entry errors.

In [ ]:
anomaly_labels, anomaly_scores = run_isolation_forest(X_train)
print(anomaly_scores.describe())


### 2.6 Save Models

In [ ]:
save_models(models_dict, '../outputs/models')

## Section 3: Phase 3 — PDP and ICE Analysis

Partial Dependence Plots (PDP) show the **average** effect of a single feature on predicted popularity, holding all other features constant. Individual Conditional Expectation (ICE) plots show this effect **per track** — diverging ICE lines reveal interaction effects where the feature's impact depends on other feature values.

We analyse RandomForest, XGBoost, and LightGBM. All three are non-linear models whose internal mechanisms are opaque — PDP/ICE provides post-hoc transparency.


### 3.1 Feature Selection

In [ ]:
X_test_sample = X_test.sample(5000, random_state=42)
features = select_pdp_features(feature_names)
print("PDP/ICE features:", features)


### 3.2 RandomForest — PDP Analysis

RandomForest is our primary model for explanation because its ensemble averaging produces stable, interpretable PDP curves.

In [ ]:
# Partial Dependence Plots — RandomForest
plot_pdp_grid(models_dict['rf'], X_test_sample, features, 'RandomForest', '../outputs/figures')


In [ ]:
# PDP/ICE interpretation table — RandomForest
rf_interp = pdp_ice_interpretation_table(models_dict['rf'], X_test_sample, features, 'RandomForest')
rf_interp


### 3.3 RandomForest — ICE Analysis

ICE plots show the effect of each feature **per track** — diverging lines reveal interaction effects.

In [ ]:
# ICE plots (standard) — RandomForest
plot_ice_grid(models_dict['rf'], X_test_sample, features, 'RandomForest', '../outputs/figures', centered=False)


In [ ]:
# ICE plots (centered) — RandomForest
plot_ice_grid(models_dict['rf'], X_test_sample, features, 'RandomForest', '../outputs/figures', centered=True)


**Interpretation — RandomForest PDP/ICE:**

- **Loudness** typically shows a monotonic positive relationship with predicted popularity — the model has learned that louder (more heavily mastered) tracks tend to chart better, consistent with the loudness war in commercial music production.
- **Energy** also tends to show a positive association — high-energy tracks dominate streaming charts in pop, hip-hop, and electronic genres.
- **Acousticness** commonly shows a negative association — highly acoustic tracks (folk, classical) receive lower predicted popularity scores on average, reflecting the dataset's genre distribution.
- **Danceability** may show non-monotonic behaviour — moderate danceability scores align with the widest range of popular genres, while extremes (very low or very high) narrow genre appeal.
- **ICE heterogeneity** for loudness and energy is often high, indicating that the effect of these features depends on other characteristics — a loud acoustic ballad behaves differently from a loud electronic track.


### 3.4 XGBoost — PDP Analysis

XGBoost is our secondary model. Its gradient-boosted trees can capture sharper feature thresholds. Comparing RF and XGBoost PDPs reveals whether the learned relationships are consistent across model architectures.

In [ ]:
# Partial Dependence Plots — XGBoost
plot_pdp_grid(models_dict['xgb'], X_test_sample, features, 'XGBoost', '../outputs/figures')


In [ ]:
# PDP/ICE interpretation table — XGBoost
xgb_interp = pdp_ice_interpretation_table(models_dict['xgb'], X_test_sample, features, 'XGBoost')
xgb_interp


### 3.5 XGBoost — ICE Analysis

In [ ]:
# ICE plots (standard) — XGBoost
plot_ice_grid(models_dict['xgb'], X_test_sample, features, 'XGBoost', '../outputs/figures', centered=False)


In [ ]:
# ICE plots (centered) — XGBoost
plot_ice_grid(models_dict['xgb'], X_test_sample, features, 'XGBoost', '../outputs/figures', centered=True)


**Interpretation — XGBoost PDP/ICE:**

- XGBoost PDPs often show **sharper transitions** than RandomForest — gradient boosting captures hard decision boundaries that manifest as step-like PDP curves, particularly for loudness and tempo.
- Where RF and XGBoost agree on the direction of an effect (e.g. both show loudness positively correlated with popularity), that finding is more robust — it is not an artefact of a single model's inductive bias.
- Discrepancies between the two models in the shape of the tempo PDP may reflect that tempo's effect on popularity is highly context-dependent (genre-mediated), making it sensitive to the model's feature interaction capacity.
- High ICE heterogeneity in XGBoost valence plots suggests the model has captured that the emotional tone of a track (valence) affects different subsets of songs differently — happy-sounding tracks benefit in pop contexts but not in hip-hop.


### 3.6 LightGBM — PDP Analysis

LightGBM is also a gradient boosting model but uses leaf-wise tree growth rather than XGBoost's level-wise approach, which can produce different decision boundaries and feature interaction patterns.

In [ ]:
# Partial Dependence Plots — LightGBM
plot_pdp_grid(models_dict['lgbm'], X_test_sample, features, 'LightGBM', '../outputs/figures')


In [ ]:
# PDP/ICE interpretation table — LightGBM
lgbm_interp = pdp_ice_interpretation_table(models_dict['lgbm'], X_test_sample, features, 'LightGBM')
lgbm_interp


### 3.7 LightGBM — ICE Analysis

In [ ]:
# ICE plots (standard) — LightGBM
plot_ice_grid(models_dict['lgbm'], X_test_sample, features, 'LightGBM', '../outputs/figures', centered=False)


In [ ]:
# ICE plots (centered) — LightGBM
plot_ice_grid(models_dict['lgbm'], X_test_sample, features, 'LightGBM', '../outputs/figures', centered=True)


**Interpretation — LightGBM PDP/ICE:**

- LightGBM's **leaf-wise growth** can produce sharper, more irregular PDP curves than XGBoost's level-wise approach — features like loudness and energy may show more abrupt transitions.
- Where all three models (RF, XGBoost, LightGBM) agree on the direction of an effect, that finding is robust across different inductive biases.
- **ICE heterogeneity** in LightGBM may differ from XGBoost on the same features — comparing the two gradient boosting models reveals whether interaction effects are a property of the data or an artefact of the specific boosting algorithm.
- LightGBM is generally faster to train than XGBoost on large datasets, making it practical for repeated explainability analyses.
